# Tugas Besar Machine Learning: Prediksi Churn Nasabah Bank
**Nama:** Fuad Muhajir Hilmi | **NPM:** 202343501787 | **Kelas:** Y7L

Notebook ini melatih model jaringan saraf (PyTorch) dan menyimpannya sebagai `churn_model.pth`.
Alur mengikuti CRISP-DM: Data Understanding, Data Preparation, Modeling, Evaluation, lalu simpan model untuk Deployment.

**Langkah:** jalankan sel dari atas ke bawah (Runtime > Run all). Sel terakhir akan mengunduh file model.

## 1. Unggah dataset
Unduh `Churn_Modelling.csv` dari Kaggle (dataset *Churn Modelling*), lalu unggah di sel di bawah.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pilih file Churn_Modelling.csv

## 2. Data Understanding

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

torch.manual_seed(42); np.random.seed(42)

df = pd.read_csv("Churn_Modelling.csv")
print(df.shape)
display(df.head())
print(df["Exited"].value_counts(normalize=True).round(3))   # proporsi nasabah yang keluar

## 3. Data Preparation

In [ ]:
FEATURES = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember",
            "EstimatedSalary", "Geography_France", "Geography_Germany", "Geography_Spain", "Gender_Male"]
NUMERIC = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]

df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])          # kolom tanpa daya prediksi
X = pd.get_dummies(df.drop(columns="Exited"), columns=["Geography", "Gender"])   # One-Hot Encoding
X = X[FEATURES].astype(float)
y = df["Exited"].astype(float).values

# 80% latih, 20% uji. Dari data latih, 15% dipakai validasi untuk memilih threshold.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42)

mean = X_tr[NUMERIC].mean().to_dict()
std = X_tr[NUMERIC].std().to_dict()
def scale(d):
    d = d.copy()
    d[NUMERIC] = (d[NUMERIC] - pd.Series(mean)) / pd.Series(std)   # statistik dari data latih saja
    return d.values.astype("float32")
Xtr, Xva, Xte = scale(X_tr), scale(X_va), scale(X_te)
print(Xtr.shape, Xva.shape, Xte.shape)

## 4. Modeling
Model utama: jaringan saraf kecil (PyTorch). Sebagai pembanding: Logistic Regression dan Random Forest.

In [ ]:
class ChurnNet(nn.Module):
    def __init__(self, n_in=len(FEATURES)):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
    def forward(self, x):
        return self.net(x)

model = ChurnNet()
opt = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCEWithLogitsLoss()
Xt, yt = torch.tensor(Xtr), torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

EPOCHS, BATCH = 60, 128
for ep in range(EPOCHS):
    perm = torch.randperm(len(Xt))
    total = 0.0
    for i in range(0, len(Xt), BATCH):
        idx = perm[i:i + BATCH]
        opt.zero_grad()
        loss = loss_fn(model(Xt[idx]), yt[idx])
        loss.backward(); opt.step()
        total += loss.item() * len(idx)
    if (ep + 1) % 10 == 0:
        print(f"epoch {ep + 1:3d}  loss {total / len(Xt):.4f}")

## 5. Evaluation
Threshold dipilih dari data validasi dengan F2-score (Recall dibuat lebih penting daripada Precision).

In [ ]:
def proba(m, X):
    m.eval()
    with torch.no_grad():
        return torch.sigmoid(m(torch.tensor(X))).numpy().ravel()

def f2(y, p, t):
    pred = (p >= t).astype(int)
    pr, rc = precision_score(y, pred, zero_division=0), recall_score(y, pred)
    return 0 if pr + rc == 0 else 5 * pr * rc / (4 * pr + rc)

p_va = proba(model, Xva)
thr = max(np.arange(0.05, 0.7, 0.01), key=lambda t: f2(y_va, p_va, t))
thr = float(round(thr, 2))

p_te = proba(model, Xte)
pred = (p_te >= thr).astype(int)
metrics = {"roc_auc": round(roc_auc_score(y_te, p_te), 3), "recall": round(recall_score(y_te, pred), 3),
           "precision": round(precision_score(y_te, pred), 3), "f1": round(f1_score(y_te, pred), 3)}
print("Threshold:", thr)
print("Metrik data uji (PyTorch):", metrics)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_te, pred))

# Pembanding
for name, clf in [("Logistic Regression", LogisticRegression(max_iter=1000)),
                  ("Random Forest", RandomForestClassifier(n_estimators=200, random_state=42))]:
    clf.fit(Xtr, y_tr)
    pp = clf.predict_proba(Xte)[:, 1]
    print(f"{name}: ROC-AUC {roc_auc_score(y_te, pp):.3f}")

## 6. Simpan model untuk Deployment
Dua file dihasilkan: `churn_model.pth` (bobot model) dan `preprocess.json` (rata-rata, std, threshold, metrik). Keduanya dibutuhkan aplikasi.

In [ ]:
torch.save(model.state_dict(), "churn_model.pth")
meta = {"features": FEATURES, "mean": mean, "std": std, "threshold": thr, "metrics": metrics}
with open("preprocess.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Tersimpan: churn_model.pth dan preprocess.json")

## 7. Unduh model
Jalankan sel ini, lalu **ambil screenshot** proses unduhnya (untuk laporan PDF).

In [ ]:
from google.colab import files
files.download("churn_model.pth")      # <- screenshot proses ini untuk laporan
files.download("preprocess.json")